# Council of Europe (eProc — Call for Tenders)

Fetches all currently live, published call-for-tenders notices from the Council of Europe's eProc portal and uploads new contracts to the unified Notion database.

**Link:** https://eproc.coe.int/callfortenders-list

**Filters applied:** no CPV or category filter, per Javiera - relatively few live opportunities on this source and most look at least somewhat relevant, so category/CPV filtering wasn't worth the added complexity (unlike the other sources). Out of the project's 29-code CPV list, this source only ever matched 1 of 25 live tenders when tested live - eProc's tenders are mostly tagged with its own internal `categoriesCoe` scheme rather than CPV, and only about half carry a CPV code at all.

**Language filter: English and Spanish only**, detected via `langid` on the title (same approach as the IDB Notion notebook, since eProc has no reliable declared-language field of its own - `language=EN` in the API request turned out to just control display language, NOT filter tender content. Confirmed live: French-titled tenders (e.g. "DGA-26-2672 CONTRAT CADRE TRAVAUX DE PLATRERIE") were coming through mislabeled as "English" before this fix). Top `langid` guess taken as-is, no confidence threshold - same convention as IDB/OppsLink.

No real free-text `description` field exists on this source's API (confirmed by inspection - the list endpoint only returns `title`, `status`, `deadline`, `publishedAt`, `categoriesCoe`, `categoriesCpv`, `countries`, `areas`, `id` and timestamps). `Description` here is built from the title plus the tender's own Council of Europe and CPV category labels - same placeholder approach as the sibling OppsLink notebook (`coe_upload_to_oppslink.ipynb`) for this source, not real free-text content. `CPV Codes` is set to "Not Disclosed" for the roughly half of tenders that don't carry one. `Client`/`Employer Website` are hardcoded to "Council of Europe" / its site homepage - single-institution portal, no separate buyer field in this response. `Contract Link` uses an **unverified** URL guess (`eproc.coe.int/callfortenders/{id}`) - same caveat as the OppsLink notebook, confirm the real permalink pattern before relying on it.

**Blocked keywords:** notices are hard-excluded if the title or description matches any term in `blocked_words.py` (shared blocklist, sources/ root) - see that file for the current list and notes on match behaviour.

**Expiry filter (added 2026-08-09):** tenders past their deadline are now skipped entirely, same check as the sibling OppsLink notebook. This notebook was previously missing that check altogether, which is why past-deadline tenders were showing up in Notion (flagged by Javiera).

**Note:** the sibling OppsLink notebook (`coe_upload_to_oppslink.ipynb`) does NOT apply this language filter - it currently uploads every live tender regardless of language, since it has no `Language` field to mislabel in the first place. Worth deciding whether OppsLink job postings should also be restricted to English/Spanish, since consultants browsing the job board may not be able to act on a French-only tender the way this Notion database (an internal tracking tool) can just log for reference.

In [1]:
import os

NOTION_TOKEN = os.environ["NOTION_TOKEN"]

DATABASE_ID = '334701e728cb8096a94cebc0985684a2'

headers_notion = {
    "Authorization": "Bearer " + NOTION_TOKEN,
    "Content-Type": "application/json",
    "Notion-Version": "2022-06-28",
}

In [2]:
# Shared keyword blocklist (sources/ root) - added 2026-08-09 per Javiera's feedback
import sys
from pathlib import Path

BLOCKED_WORDS_PATH = Path("../blocked_words.py")
if not BLOCKED_WORDS_PATH.exists():
    raise FileNotFoundError(f"Could not find {BLOCKED_WORDS_PATH.resolve()}")

sys.path.insert(0, str(BLOCKED_WORDS_PATH.parent.resolve()))
from blocked_words import is_blocked, blocked_keyword_hits


In [3]:
import requests
import pandas as pd
import json
import html
import os
import time
import langid
from datetime import datetime, timezone
from dateutil import parser as _dateparser

BASE_URL = "https://eproc.coe.int/api/CallForTenders/filtered-public-paginated"
COE_SITE_URL = "https://eproc.coe.int"

EMPTY_RESULT = {"result": [], "nbTotalElements": None, "nbTotalFilteredElements": 0}

# Only these two languages pass the filter - same convention as IDB's Notion notebook.
SUPPORTED_COE_LANGUAGES = {"en", "es"}
LANG_DISPLAY_NAME = {"en": "English", "es": "Spanish"}


def clean_description(description):
    if not description:
        return "Not Disclosed"
    cleaned = html.unescape(description)
    cleaned = cleaned.replace('\r\n', ' ').replace('\n', ' ')
    return ' '.join(cleaned.split()).strip()


def detect_coe_language(title: str):
    """Top langid guess, taken as-is - same approach as IDB's Notion notebook."""
    try:
        lang, _ = langid.classify(title)
        return lang
    except Exception:
        return None


def is_coe_expired(deadline_iso: str) -> bool:
    """
    Same check as the sibling OppsLink notebook (coe_upload_to_oppslink.ipynb) - this
    notebook was previously missing it entirely, which is why past-deadline tenders were
    getting uploaded (flagged by Javiera, 2026-08-09).
    """
    if not deadline_iso:
        return False
    try:
        dt = _dateparser.parse(deadline_iso)
        if not dt:
            return False
        return dt.date() < datetime.now(timezone.utc).date()
    except Exception:
        return False


def fetch_coe_page(offset: int, limit: int = 25) -> dict:
    resp = requests.get(BASE_URL, params={
        "offset": offset,
        "limit": limit,
        "status": "Published",
        "language": "EN",
        "orderByParam": "Status",
        "orderByOption": "Ascending",
        # deliberately no categoryCpvId / categoryCoeId param - Javiera wants everything live, unfiltered by category
    })
    if resp.status_code == 204:
        return EMPTY_RESULT  # zero live tenders - legitimate response, not an error
    resp.raise_for_status()
    return resp.json()


def format_categories(raw_categories) -> tuple[str, str]:
    """
    categoriesCoe / categoriesCpv both come back as a list of dicts with
    'code' and 'description'. Returns (codes_joined, descriptions_joined),
    each de-duplicated and comma-separated, or ("", "") if the list is empty
    (categoriesCpv is empty for roughly half of CoE's live tenders).
    """
    if not raw_categories:
        return "", ""
    codes, descs = [], []
    for cat in raw_categories:
        c = (cat.get("code") or "").strip()
        d = (cat.get("description") or "").strip()
        if c and c not in codes:
            codes.append(c)
        if d and d not in descs:
            descs.append(d)
    return ", ".join(codes), ", ".join(descs)

In [4]:
# 1) Load already-uploaded titles to avoid duplicates
csv_path = "coe_contract_titles.csv"
existing_titles = set()
if os.path.exists(csv_path):
    try:
        prev = pd.read_csv(csv_path)
        if "Title" in prev.columns:
            existing_titles = set(prev["Title"].dropna().astype(str).str.strip().str.lower())
    except Exception as e:
        print("Warning: could not read", csv_path, ":", e)

# 2) Fetch everything currently live/published - no CPV or category filter, per Javiera
EFFECTIVE_LIMIT = 25  # confirmed safe via live testing; raising this above 25 has NOT been verified

first_page = fetch_coe_page(offset=0, limit=EFFECTIVE_LIMIT)
total_live = first_page["nbTotalFilteredElements"]
print(f"{total_live} live published tenders on Council of Europe's site right now.")

all_coe_raw = list(first_page["result"])
offset = EFFECTIVE_LIMIT
while offset < total_live:
    page = fetch_coe_page(offset=offset, limit=EFFECTIVE_LIMIT)
    all_coe_raw.extend(page["result"])
    offset += EFFECTIVE_LIMIT
    time.sleep(1)

print(f"Fetched {len(all_coe_raw)} raw tenders (expected {total_live})")
if len(all_coe_raw) != total_live:
    print("⚠️ Mismatch between fetched count and reported total - investigate before uploading.")

# 3) Dedup + language filter (English/Spanish only) + parse
# ⚠️ See README cell at the top for the known gaps this section works around:
# no confirmed `description` field, no confirmed tender permalink, and the
# single hardcoded "Council of Europe" employer.

_warned_about_description_gap = False
_warned_about_link_gap = False

extracted_data = []
skipped_unsupported_language = 0
skipped_expired = 0

for item in all_coe_raw:
    title = (item.get("title") or "").strip()
    if not title:
        continue

    if title.strip().lower() in existing_titles:
        continue

    lang = detect_coe_language(title)
    if lang not in SUPPORTED_COE_LANGUAGES:
        skipped_unsupported_language += 1
        print(f"🌐 Skipping unsupported language ({lang}): {title}")
        continue

    tender_id = item.get("id")
    deadline_iso = item.get("deadline")  # e.g. "2026-08-13T17:00:00+02:00"

    if is_coe_expired(deadline_iso):
        skipped_expired += 1
        print(f"🚫 Skipping expired: {title} (deadline {deadline_iso})")
        continue

    coe_codes, coe_descs = format_categories(item.get("categoriesCoe"))
    cpv_codes, cpv_descs = format_categories(item.get("categoriesCpv"))

    countries = item.get("countries") or []
    country_names = [c.get("name") for c in countries if c.get("name")]
    location = ", ".join(country_names) if country_names else "Council of Europe (all member states)"

    # No real free-text description field exists on this endpoint (confirmed by
    # inspection - see README). Built from the tender's own category labels as
    # a placeholder, same approach as the sibling OppsLink notebook.
    description_parts = [title]
    if coe_descs:
        description_parts.append(f"Council of Europe categories: {coe_descs}.")
    if cpv_descs:
        description_parts.append(f"CPV categories: {cpv_descs}.")
    description = clean_description(" ".join(description_parts))

    if not _warned_about_description_gap:
        print("⚠️ Using title + category labels as a description placeholder for all CoE contracts - "
              "no real free-text description field was found on this endpoint. See README.")
        _warned_about_description_gap = True

    # UNVERIFIED URL pattern - confirm the real one by opening a live tender in
    # the browser and comparing, then fix this in one place.
    link = f"{COE_SITE_URL}/callfortenders/{tender_id}" if tender_id is not None else COE_SITE_URL
    if not _warned_about_link_gap:
        print(f"⚠️ Tender permalink pattern is UNVERIFIED (e.g. {link}) - confirm the real URL before production use.")
        _warned_about_link_gap = True

    if is_blocked(title, description):
        hits = blocked_keyword_hits(title, description)
        print(f"⛔ Skipping blocked keyword ({', '.join(hits)}): {title}")
        continue

    extracted_data.append({
        "closing_date": deadline_iso,
        "country": location,
        "client": "Council of Europe",  # single-institution portal - no separate buyer field in this response
        "client_link": COE_SITE_URL,
        "link": link,
        "title": title,
        "description": description,
        "value": "Unavailable",           # no value field on this endpoint
        "cpv_codes": cpv_codes or "Not Disclosed",
        "language": LANG_DISPLAY_NAME.get(lang, lang),
    })

print(f"✅ {len(extracted_data)} new contracts ready for Notion upload "
      f"(skipped {skipped_unsupported_language} non-English/Spanish, {skipped_expired} expired)")

20 live published tenders on Council of Europe's site right now.
Fetched 20 raw tenders (expected 20)


🌐 Skipping unsupported language (fr): 2026AO21 - Achat chambre froide extérieure -35°C
🌐 Skipping unsupported language (fr): DGS-26-2991 - Contrat cadre objets de visibilité
🌐 Skipping unsupported language (fr): DGS-26-1043_Rénovation globale du bâtiment D_Lot 05 : PIERRE NATURELLE V2
🌐 Skipping unsupported language (fr): DGS-26-1050_Rénovation globale du bâtiment D_Lot 12 : CARRELAGE - FAIENCE V2
🌐 Skipping unsupported language (fr): 2026MC49814930 - Acquisition d'un polarimètre numérique haute précision pour analyses de routine
🌐 Skipping unsupported language (fr): DGS-26-2945 - Achat de Carrés de soie
🌐 Skipping unsupported language (fr): 2026MC28981308 – Fourniture d’articles promotionnels de visibilité
✅ 0 new contracts ready for Notion upload (skipped 7 non-English/Spanish, 0 expired)


### Upload to Notion

In [5]:
def create_page(properties: dict) -> bool:
    url = "https://api.notion.com/v1/pages"
    payload = {"parent": {"database_id": DATABASE_ID}, "properties": properties}
    res = requests.post(url, headers=headers_notion, json=payload, timeout=60)
    if not res.ok:
        print("❌ Notion error:", res.status_code, res.text[:500])
        return False
    print(f"✅ Page created: {properties['Name']['title'][0]['text']['content']}")
    return True


def _safe_str(x, default=""):
    if x is None:
        return default
    s = str(x).strip()
    return s if s else default


def _safe_iso(dt_str):
    if not dt_str:
        return None
    try:
        parsed = _dateparser.parse(dt_str)
        return parsed.isoformat()
    except Exception:
        return None


now_iso = datetime.now(timezone.utc).isoformat()
new_titles_for_csv = []

# Upload newest first, matching the other Notion notebooks' convention
for contract in list(reversed(extracted_data)):
    name = _safe_str(contract.get("title"))[:1000]
    if not name:
        continue

    closing_date_iso = _safe_iso(contract.get("closing_date"))

    props = {
        "Name": {"title": [{"text": {"content": name}}]},
        "CPV Codes": {"rich_text": [{"text": {"content": _safe_str(contract.get("cpv_codes"))[:2000]}}]},
        "Client": {"rich_text": [{"text": {"content": _safe_str(contract.get("client"), "Not Disclosed")[:2000]}}]},
        "Contract Link": {"url": _safe_str(contract.get("link")) or None},
        "Date Added": {"date": {"start": now_iso, "end": None}},
        "Closing Date": {"date": {"start": closing_date_iso, "end": None}} if closing_date_iso else {"date": None},
        "Description": {"rich_text": [{"text": {"content": _safe_str(contract.get("description"), "Not Disclosed")[:2000]}}]},
        "Employer Website": {"url": _safe_str(contract.get("client_link")) or None},
        "Language": {"rich_text": [{"text": {"content": _safe_str(contract.get("language"))[:2000]}}]},
        "Location": {"rich_text": [{"text": {"content": _safe_str(contract.get("country"))[:2000]}}]},
        "Reviewed By": {"select": {"name": "N/A"}},
        "Review Status": {"select": {"name": "Not Reviewed"}},
        "Value": {"rich_text": [{"text": {"content": _safe_str(contract.get("value"), "Unavailable")[:2000]}}]},
        "Contract Status": {"select": {"name": "Open"}},
        "Source": {"select": {"name": "Council of Europe"}},
    }

    try:
        success = create_page(props)
        if success:
            new_titles_for_csv.append({"Title": name})
    except Exception as e:
        print(f"Error creating Notion page for '{name}': {e}")

# Save newly uploaded titles to CSV for next run's dedup
if new_titles_for_csv:
    new_df = pd.DataFrame(new_titles_for_csv, columns=["Title"])
    header_needed = not os.path.exists(csv_path)
    new_df.to_csv(csv_path, mode="a", header=header_needed, index=False)

print(f"✅ Uploaded {len(new_titles_for_csv)} new Council of Europe contracts to Notion.")

✅ Uploaded 0 new Council of Europe contracts to Notion.
